# Audit de données, variables et classes déséquilibrées

**Objectifs.** Repérer doublons et trous, quantifier une valeur atypique sans la supprimer automatiquement, créer une feature explicable, préserver le ratio des classes et comparer les modèles avec une métrique adaptée. Durée : 90 min.

## 1. Un jeu synthétique à auditer

Les observations sont générées artificiellement. Nous injectons des doublons, quelques valeurs manquantes et une valeur extrême pour pratiquer les contrôles. La cible ne doit pas être utilisée comme feature.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from learning_hub.preprocessing import make_churn_data

X, y = make_churn_data(n_samples=800, random_state=24)
raw = X.assign(churn=y)
raw = pd.concat([raw, raw.iloc[:3]], ignore_index=True)
raw.loc[raw.index[12:30:4], 'monthly_charge'] = np.nan
raw.loc[40, 'monthly_charge'] = 650.0
print('shape:', raw.shape)
print('duplicates:', raw.duplicated().sum())
print('missing:\n', raw.isna().sum())
print('target balance:\n', raw.churn.value_counts(normalize=True).round(3))

### Deux questions d'audit différentes

Le taux manquant concerne la **qualité des features** ; la proportion positive concerne la **distribution de la cible**. Les deux influencent le protocole de modèle et de métrique. Ces graphiques donnent des comptes et des proportions ; ils ne disent pas si les données représentent un vrai service.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
raw.isna().sum().plot(kind='bar', ax=axes[0], color='tab:orange')
axes[0].set(title='Valeurs absentes par feature', ylabel='Nombre de lignes')
raw['churn'].value_counts(normalize=True).sort_index().plot(kind='bar', ax=axes[1], color=['tab:blue', 'tab:red'])
axes[1].set(title='Proportion des classes', xlabel='Classe synthétique', ylabel='Part')
plt.setp(axes[0].get_xticklabels(), rotation=35, ha='right')
plt.tight_layout(); plt.show()

## 2. Nettoyer avant le split et décider des outliers

Les lignes dupliquées peuvent traverser train et test si on split avant déduplication. Ici nous enlevons uniquement des lignes strictement identiques et gardons le premier exemplaire. Une valeur extrême est un signal à examiner : elle peut être vraie, erronée ou issue d’une autre unité.

In [ ]:
clean = raw.drop_duplicates().copy()
print('rows removed:', len(raw) - len(clean))
q1, q3 = clean['monthly_charge'].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = clean.loc[(clean.monthly_charge < lower) | (clean.monthly_charge > upper)]
print('IQR review bounds:', round(lower, 1), round(upper, 1), '| flagged rows:', len(outliers))
clean['monthly_charge'].plot(kind='box', title='Valeurs mensuelles simulées'); plt.show()

## 3. Split stratifié et feature engineering

On garde la cible à part, puis on sépare de manière stratifiée. Une transformation déterministe de colonnes déjà disponibles peut être créée avant split ; les paramètres appris (médiane, moyenne, variance, vocabulaire) restent dans la pipeline. `calls_per_month` convertit les appels observés sur 90 jours en taux mensuel.

In [ ]:
from sklearn.model_selection import train_test_split

labels = clean.pop('churn')
clean['calls_per_month'] = clean['support_calls_90d'] / 3.0
x_train, x_test, y_train, y_test = train_test_split(clean, labels, stratify=labels, test_size=0.25, random_state=24)
print('positive fractions train/test:', round(y_train.mean(), 3), round(y_test.mean(), 3))
print('feature columns:', list(x_train.columns))

### Pourquoi stratifier ?

Si une classe représente 20 % des exemples, un petit split purement aléatoire peut accidentellement en mettre beaucoup moins dans le test. Le split stratifié maintient approximativement la proportion de chaque classe. La séparation correcte dépend toutefois du phénomène : si les mêmes clients réapparaissent, il faut séparer par client ; si le modèle prédit le futur, il faut considérer un split temporel.

In [ ]:
train_share = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_share = pd.Series(y_test).value_counts(normalize=True).sort_index()
comparison = pd.DataFrame({'train': train_share, 'test': test_share})
ax = comparison.plot(kind='bar', figsize=(5.5, 3.5), ylim=(0, 1), rot=0)
ax.set(xlabel='Classe', ylabel='Proportion', title='Proportions après split stratifié')
plt.show()

## 4. Preprocessing, baseline et déséquilibre

Accuracy peut être élevée en prédisant toujours la classe majoritaire. Average Precision mesure le compromis precision–recall sur plusieurs seuils. Toutes les transformations et la sélection de variables sont ajustées dans le pipeline à l’intérieur des folds.

### Feature selection dans le fold

`SelectKBest(f_classif, k=5)` garde les cinq colonnes transformées dont l'association statistique avec la cible est la plus forte sur le train du fold. L'encodeur peut augmenter le nombre de colonnes ; `k=5` désigne donc des colonnes **après** transformation. La sélection est dans la pipeline pour éviter d'apprendre depuis le fold de validation. Une variable associée n'est pas nécessairement causale ou disponible à l'inférence.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from learning_hub.preprocessing import make_tabular_preprocessor

numeric = ['tenure_months', 'monthly_charge', 'support_calls_90d', 'calls_per_month']
categorical = ['contract']
preprocess = make_tabular_preprocessor(numeric, categorical)
baseline = DummyClassifier(strategy='most_frequent').fit(x_train, y_train)
model = Pipeline([('preprocess', preprocess), ('select', SelectKBest(f_classif, k=5)), ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))])
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=24)
scores = cross_val_score(model, x_train, y_train, cv=folds, scoring='average_precision')
model.fit(x_train, y_train)
probability = model.predict_proba(x_test)[:, 1]
print('majority baseline accuracy:', round(baseline.score(x_test, y_test), 3))
print('CV average precision mean/std:', round(scores.mean(), 3), round(scores.std(), 3))
print(classification_report(y_test, probability >= 0.5, zero_division=0))
print('test average precision:', round(average_precision_score(y_test, probability), 3))

## 5. Interprétation, leakage et limites

IQR bounds calculés sur toute la table sont descriptifs ; pour clipper en production, borner à partir du train et appliquer exactement la même règle à validation/test et futures données. La déduplication dépend de l’identifiant métier et du moment de collecte. La feature `calls_per_month` n’ajoute aucune nouvelle observation ; elle réexprime la mesure. Les labels synthétiques ne représentent aucun vrai client.

## Suite et exercices

Évaluez l’effet d’une autre règle de missingness, comparez `class_weight=None` à `balanced`, inspectez les faux négatifs, et séparez par client si plusieurs lignes décrivent le même client. Ne choisissez pas le seuil avec le test. Corrélations et importances ne prouvent pas de causalité.